In [ ]:
# pip install langgraph langchain-anthropic langchain-experimental
# pip install pandas scikit-learn openpyxl matplotlib

In [ ]:
from langchain.agents import create_agent
from langchain_experimental.tools import PythonREPLTool
from langchain_openai import ChatOpenAI

In [ ]:
api_key = ""   # Replace with your actual OpenAI API key

llm = ChatOpenAI(api_key=api_key, model="gpt-5.4-mini-2026-03-17")  # or model="gpt-4"

In [ ]:
tools = [PythonREPLTool()]

agent = create_agent(llm, tools)

In [ ]:
result = agent.invoke({
    "messages": [
        {
            "role": "system",
            "content": """You are a data science assistant with access to a Python REPL.
                            When asked to build a model, always:
                            1. Load and inspect the data first
                            2. Clean and prepare it
                            3. Build the model
                            4. Print clear results and metrics
                            5. Save any plots as PNG files

                            The file stock_prices.xlsx is in the current working directory.
                            Always use the columns 'Date' and 'Close'.
                            """
        },
        {
            "role": "user",
            "content": """
            Load stock_prices.xlsx and build a price prediction model using linear regression based on prices of the last few months"
            lots actual vs predicted prices
            """
        }
    ]
})

print(result["messages"][-1].content)

Writing new code

In [ ]:
base_repl = PythonREPLTool()

@tool
def safe_python_repl(code: str) -> str:
    """
    Use this tool to write and execute Python code.
    Use it for calculations, simulations, data analysis, or anything that requires running code.
    Input should be valid Python code as a string.
    """
    print("\n--- CODE TO BE EXECUTED ---")
    print(code)
    print("---------------------------")
    approval = input("Approve execution? (yes/no): ").strip().lower()
    if approval == "yes":
        return base_repl.run(code)
    return "Execution rejected by user."

In [11]:
# Tool: The latter provides a human in the loop

python_tool = PythonREPLTool()
#python_tool = safe_python_repl

# Agent

tools = [python_tool]

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="You are a helpful assistant that can write and execute Python code.",
)

# Run
from IPython.display import Markdown, display


task = """
Simulate the motion of a ball dropped from 100 meters height under gravity (g = 9.81 m/s^2).
Write a Python function that computes height after time t.
Then calculate the height at t=0, 1, 2, 3, 4, and 5 seconds.
Additionally provide the code used
"""


response = agent.invoke({"messages": [{"role": "user", "content": task}]})
display(Markdown(response["messages"][-1].content))



For a ball dropped from rest from height \(h_0 = 100\) m under gravity \(g = 9.81\ \text{m/s}^2\), the height after time \(t\) is:

\[
h(t) = 100 - \frac{1}{2}gt^2 = 100 - 4.905t^2
\]

### Python function

```python
def height_after_time(t, h0=100, g=9.81):
    return h0 - 0.5 * g * t**2
```

### Heights at specific times

Using the function:

- \(t = 0\) s → \(100.000\) m
- \(t = 1\) s → \(95.095\) m
- \(t = 2\) s → \(80.380\) m
- \(t = 3\) s → \(55.855\) m
- \(t = 4\) s → \(21.520\) m
- \(t = 5\) s → \(-22.625\) m

### Note
Physically, the ball hits the ground before 5 seconds.  
The impact time is when \(h(t)=0\):

\[
100 - 4.905t^2 = 0 \Rightarrow t \approx 4.52\ \text{s}
\]

So after about 4.52 seconds, the height would no longer be valid unless you model the bounce or stop at ground level.

### Code used

```python
def height_after_time(t, h0=100, g=9.81):
    return h0 - 0.5 * g * t**2

times = [0, 1, 2, 3, 4, 5]

for t in times:
    h = height_after_time(t)
    print(f"t = {t} s, height = {h:.3f} m")
```

If you want, I can also provide a version that stops the height at 0 m once the ball hits the ground.